+ pydantic
+ TypedDict
+ JSON Schema
+ dataclass

pydantic 会抛异常，功能丰富，且返回Schema类实例，其他不抛异常且返回为字典

旧模型只能通过prompt要求llm输出格式化，且还得用json.load转换

In [ ]:

import base64
from langchain_deepseek import ChatDeepSeek
from langchain.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
class LLM:
    def __init__(self):
        load_dotenv(".env.local",override=True)
        self.api_key= os.getenv("DEEPSEEK_API_KEY").strip()
        self.model= os.getenv("MODEL").strip()
        self.url= os.getenv("DEEPSEEK_BASE_URL").strip()
        try:
            self.thinking_timeout= float(os.getenv("TINKING_TIMEOUT",30).strip())
            self.num_retries= int(os.getenv("RETRY_NUM",2).strip())
            self.temperature= int(os.getenv("TEMPERATURE",1).strip())
            self.stream= True if os.getenv("STREAM").strip() == 'True' else False
            self.max_token= int(os.getenv("MAX_TOKEN").strip()) if os.getenv("MAX_TOKEN").strip() else None
        except Exception as e:
            raise Exception(f'配置文件传入非法参数。错误信息：{e}')
        self.set_llm()
    def set_llm(self):
        self.llm = ChatDeepSeek(
            model= self.model,
            api_key= self.api_key,
            streaming= self.stream,
            api_base= self.url,
            temperature=self.temperature,
            request_timeout= self.thinking_timeout,
            max_tokens= self.max_token,
            max_retries= self.num_retries,
            extra_body={"thinking":{"type":"disabled"}}## enabled时，会无法pydantic格式化，实际是enabled时，工具调用tool_choice无法设置required
            # model_kwargs=   {'tools':[]}##用来存放一些langchain没有列出但模型本身支持的，比如tools
            # configurable_fields= ('model','temperature') ## 用来允许 config中的configurable 覆盖
        )

model = LLM().llm

In [12]:
#pydantic--实际推荐
from pydantic import BaseModel,Field
class Person(BaseModel):
    """人物信息"""
    name : str = Field(
        description='姓名'
    )
    age : int = Field(
        description='年龄'
    )
    occupation : str= Field(
        description='职业'
    )
    gender : str= Field(
        description='性别'
    )
structured_model = model.with_structured_output(Person)
res = structured_model.invoke("给我一个真实的示例员工信息")
print(res)

name='张伟' age=32 occupation='软件工程师' gender='男'
